# Shesha Bio Tutorial: Split-Half Reproducibility

This notebook demonstrates `shesha.bio.split_half_reproducibility` — a direct assay of
effect-direction reproducibility for single-cell CRISPR perturbation experiments.

**What you'll learn:**
- What split-half reproducibility measures and why it matters
- How to compute it for every perturbation in a dataset
- How to use `magnitude_matched_comparison` to control for the SNR confound
- How to interpret the results biologically

**Requirements:**
```bash
pip install "shesha-geometry[bio]" pertpy scanpy
```

The `[bio]` extra installs `anndata` and `scikit-learn`, which are required by `shesha.bio`.

> **Note (Google Colab):** The Norman 2019 dataset (~111k cells) requires ~12 GB RAM.
> Free-tier Colab may hit out-of-memory errors; Colab Pro or a local machine is recommended.

### References
  **Norman et al. (2019).** *Science*. "Exploring genetic interaction manifolds constructed
  from rich single-cell phenotypes." [10.1126/science.aax4438](https://doi.org/10.1126/science.aax4438)

## 1. Setup and Imports

In [ ]:
# Install dependencies (run once)
!pip install "shesha-geometry[bio]" pertpy scanpy -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

# Shesha
from shesha.bio import (
    compute_stability,
    compute_magnitude,
    split_half_reproducibility,
    magnitude_matched_comparison,
)

# Single-cell tools
import scanpy as sc
import pertpy as pt
from anndata import AnnData

SEED = 320
np.random.seed(SEED)

print("Imports successful!")

## 2. Background: What is Split-Half Reproducibility?

A perturbation has high **stability** (Sp) when individual cells all shift in the same
direction relative to the control centroid. But stability is measured on the full cell
population — it doesn't directly answer:

> *If I reran this perturbation on a fresh sample of cells, would I see the same effect?*

**Split-half reproducibility** answers this directly:

1. For each perturbation, randomly split its cells 50/50 into two independent halves.
2. Compute the mean shift vector of each half relative to the control centroid.
3. Measure the **cosine similarity** between the two shift vectors.
4. Repeat N times and average.

A **high cosine** (near 1) means both halves point in the same direction — the effect is
reproducible. A **low cosine** (near 0) means the two halves disagree — the effect is noisy
or cell-state dependent.

**Key distinction from stability (Sp):**
- Sp measures within-population consistency (are *these* cells coherent?)
- Split-half measures held-out reproducibility (do *independent subsamples* agree?)

**Why control for magnitude?**  
Stronger perturbations have higher SNR, so they *trivially* appear more reproducible.
`magnitude_matched_comparison` bins perturbations by effect size and tests whether
high-Sp perturbations are *still* more reproducible within each magnitude bin.

## 3. Load and Preprocess the Norman 2019 Dataset

In [ ]:
print("Loading Norman 2019 dataset...")
adata = pt.dt.norman_2019()
print(f"Dataset: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes")
print(f"Perturbation column: perturbation_name")
print(f"Unique perturbations: {adata.obs['perturbation_name'].nunique()}")

In [ ]:
# Standard scanpy preprocessing
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=True)
sc.pp.pca(adata, n_comps=50, random_state=SEED)

# Build a PCA-space AnnData for Shesha
# All shesha.bio functions operate on .X, so we expose PCA coords as .X
adata_pca = AnnData(X=adata.obsm['X_pca'], obs=adata.obs)

print(f"PCA AnnData: {adata_pca.shape[0]:,} cells x {adata_pca.shape[1]} components")

## 4. Compute Split-Half Reproducibility

`split_half_reproducibility` returns a DataFrame indexed by perturbation name with:
- `split_half_cosine` — mean cosine similarity across N splits
- `n_cells` — number of cells for that perturbation

In [ ]:
print("Computing split-half reproducibility (n_splits=50)...")

repro = split_half_reproducibility(
    adata_pca,
    perturbation_key="perturbation_name",
    control_label="control",
    n_splits=50,
    random_state=SEED,
    min_cells=30,
)

print(f"Computed for {len(repro)} perturbations")
print()
print("Top 10 most reproducible perturbations:")
print(repro.sort_values('split_half_cosine', ascending=False).head(10).to_string())

In [ ]:
print("Bottom 10 least reproducible perturbations:")
print(repro.sort_values('split_half_cosine').head(10).to_string())

## 5. Compute Stability (Sp) and Magnitude (Mp)

To run `magnitude_matched_comparison` and test whether Sp predicts reproducibility
*beyond* magnitude, we also need stability and effect size scores.

In [ ]:
print("Computing stability (Sp) and magnitude (Mp)...")

stability_scores = compute_stability(
    adata_pca,
    perturbation_key='perturbation_name',
    control_label='control',
)

magnitude_scores = compute_magnitude(
    adata_pca,
    perturbation_key='perturbation_name',
    control_label='control',
)

# Combine into one DataFrame
df_sp = pd.DataFrame({
    'Sp': stability_scores,
    'Mp': magnitude_scores,
})

# Join with reproducibility (drop duplicate n_cells from repro)
df = df_sp.join(repro.drop(columns=['n_cells']), how='inner')
df = df.dropna(subset=['Sp', 'Mp', 'split_half_cosine'])

print(f"Final dataset: {len(df)} perturbations with all three metrics")
df.head()

## 6. Does Stability Predict Reproducibility?

We test three correlations:
- **Sp vs split-half cosine** (primary claim)
- **Mp vs split-half cosine** (magnitude confound)
- **Sp vs Mp** (how correlated are the two predictors?)

In [ ]:
rho_sp, p_sp = spearmanr(df['Sp'], df['split_half_cosine'])
rho_mp, p_mp = spearmanr(df['Mp'], df['split_half_cosine'])
rho_sp_mp, _ = spearmanr(df['Sp'], df['Mp'])

print("Spearman correlations vs split-half cosine:")
print(f"  Stability (Sp):  rho = {rho_sp:+.3f}   p = {p_sp:.2e}")
print(f"  Magnitude (Mp):  rho = {rho_mp:+.3f}   p = {p_mp:.2e}")
print()
print(f"Sp vs Mp correlation: rho = {rho_sp_mp:+.3f}  (confound check)")

## 7. Magnitude-Matched Comparison

`magnitude_matched_comparison` bins perturbations into magnitude quartiles, then within
each bin splits them at the Sp median and compares mean split-half cosine.

If high-Sp perturbations are more reproducible in **every** magnitude bin, Sp predicts
reproducibility independently of effect size.

In [ ]:
bins = magnitude_matched_comparison(
    df,
    stability_col='Sp',
    repro_col='split_half_cosine',
    magnitude_col='Mp',
    n_bins=4,
)

print("Magnitude-matched comparison (high-Sp vs low-Sp within each quartile):")
print()
display_cols = ['mag_bin', 'n', 'mag_min', 'mag_max',
                'high_stability_mean', 'low_stability_mean',
                'difference', 'within_bin_rho', 'within_bin_pvalue']
print(bins[display_cols].to_string(index=False, float_format='{:.4f}'.format))

print()
n_pos = (bins['difference'] > 0).sum()
if n_pos == len(bins):
    print(">>> CONSISTENT: high-Sp has higher split-half cosine in ALL magnitude bins")
else:
    print(f">>> high-Sp has higher split-half cosine in {n_pos}/{len(bins)} magnitude bins")

## 8. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
sns.set_style('whitegrid')

# ── Panel A: Sp vs split-half cosine ────────────────────────
ax = axes[0]
ax.scatter(df['Sp'], df['split_half_cosine'],
           alpha=0.3, s=15, c='steelblue', label='All perturbations')

highlights = {
    'KLF1':  ('#d62728', 'KLF1 (lineage driver)'),
    'CEBPA': ('#2ca02c', 'CEBPA (pleiotropic)'),
    'SPI1':  ('#ff7f0e', 'SPI1'),
}
for gene, (color, label) in highlights.items():
    sub = df[df.index.str.contains(f'^{gene}$')]
    if len(sub):
        ax.scatter(sub['Sp'], sub['split_half_cosine'],
                   c=color, s=70, zorder=5, label=label, edgecolors='white')

ax.set_xlabel('Stability (Sp)', fontsize=11)
ax.set_ylabel('Split-Half Cosine', fontsize=11)
ax.set_title(f'Sp vs Reproducibility\nρ = {rho_sp:.3f}  p = {p_sp:.1e}', fontsize=11)
ax.legend(fontsize=8)

# ── Panel B: Mp vs split-half cosine ────────────────────────
ax = axes[1]
ax.scatter(df['Mp'], df['split_half_cosine'],
           alpha=0.3, s=15, c='darkorange')
ax.set_xlabel('Magnitude (Mp)', fontsize=11)
ax.set_ylabel('Split-Half Cosine', fontsize=11)
ax.set_title(f'Mp vs Reproducibility\nρ = {rho_mp:.3f}  p = {p_mp:.1e}', fontsize=11)

# ── Panel C: magnitude-matched bar chart ────────────────────
ax = axes[2]
x = np.arange(len(bins))
w = 0.35
ax.bar(x - w/2, bins['high_stability_mean'], w,
       label='High Sp', color='steelblue', alpha=0.85)
ax.bar(x + w/2, bins['low_stability_mean'], w,
       label='Low Sp', color='lightcoral', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(bins['mag_bin'])
ax.set_xlabel('Magnitude Quartile', fontsize=11)
ax.set_ylabel('Mean Split-Half Cosine', fontsize=11)
ax.set_title('Magnitude-Matched Comparison\n(High vs Low Sp within each bin)', fontsize=11)
ax.legend()

plt.tight_layout()
plt.savefig('split_half_norman.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved -> split_half_norman.png")

## 9. Interpretation

### Panel A — Sp vs Reproducibility
A positive correlation means perturbations with **higher geometric stability also produce
more reproducible directional effects** when you hold out half the cells.

- **KLF1** (top-right): high Sp, high reproducibility — a tight lineage-specific program
  that pushes all cells in the same direction.
- **CEBPA** (lower-right): strong effect but more variable — a pioneer factor whose impact
  depends on local chromatin state.

### Panel B — Mp vs Reproducibility
Magnitude also correlates with reproducibility (SNR confound: bigger effects are easier
to detect in each half). This is why controlling for magnitude matters.

### Panel C — Magnitude-Matched Comparison
This is the key control. Within each magnitude quartile, **high-Sp perturbations still
show higher split-half cosine than low-Sp ones**. This means Sp captures genuine
directional coherence *beyond* what effect size alone explains.

**Practical implication:** Sp can be used to prioritise hits in CRISPR screens — a
high-Sp perturbation is more likely to replicate in an independent experiment, even when
its raw effect size is similar to a low-Sp perturbation.

---

### Quick Reference

```python
from shesha.bio import split_half_reproducibility, magnitude_matched_comparison

# Step 1: compute reproducibility
repro = split_half_reproducibility(
    adata_pca,
    perturbation_key="perturbation_name",
    control_label="control",
    n_splits=50,
    random_state=320,
)

# Step 2: magnitude-matched comparison (needs Sp and Mp columns)
bins = magnitude_matched_comparison(
    df,                        # DataFrame with Sp, Mp, split_half_cosine
    stability_col="Sp",
    repro_col="split_half_cosine",
    magnitude_col="Mp",
    n_bins=4,
)
```

| split_half_cosine | Interpretation |
|---|---|
| > 0.95 | Highly reproducible — strong, coherent directional effect |
| 0.7 – 0.95 | Reproducible — reliable hit |
| 0.3 – 0.7 | Moderate — effect present but noisy or cell-state dependent |
| < 0.3 | Low reproducibility — likely heterogeneous or weak effect |